# Wearable AI for Diabetes — D1NAMO Phase 2: Non-Invasive Glucose Prediction

**B.Tech Final Year Project | Biomedical AI & Wearable Sensing**

### Project Objective
Phase 2 explores whether continuous glucose levels, trends, and clinical ranges can be estimated **entirely non-invasively without using any past glucose history**.

### Inputs (Zero Glucose History)
- Heart Rate (`HR_unified` from BioHarness & ECG)
- Raw ECG-derived Heart Rate Variability (**SDNN**, **RMSSD**, **pNN50**)
- Physical Activity & Accelerometer telemetry
- Device Temperature (`DeviceTemp`)
- Breathing Rate (`BR`)
- Sinusoidal Circadian Encodings (`hour_sin`, `hour_cos`, `sin_2phi`, `cos_2phi`)
- Rest/Sleep Proxy (`is_night`)

### Clinical Tasks
1. **Task A: Glucose Trend Prediction (3-Class):** Predict whether glucose will **Increase**, **Decrease**, or remain **Stable** over 30 min and 60 min ($\pm 1.0$ mg/dL/min clinical rate threshold).
2. **Task B: Glycemic Range Classification (3-Class):** Classify current metabolic state into **Hypoglycemic** ($<70$ mg/dL), **Target / Normal** ($70-180$ mg/dL), or **Hyperglycemic** ($>180$ mg/dL) using international consensus Time-in-Range boundaries.

### Evaluation Methodology
Evaluated via **Leave-One-Subject-Out (LOSO) Cross-Validation** across all 9 subjects using Balanced Accuracy, Macro F1-Score, Confusion Matrices, and Feature Importance Analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries loaded successfully!')

## Step 1 — Load Non-Invasive Multimodal Dataset
We verify that zero glucose history or lag features are included in the feature matrix.

In [ ]:
data_path = 'data/d1namo_multimodal_master.parquet'
if not os.path.exists(data_path):
    data_path = 'data/d1namo_multimodal_master.csv'
    df = pd.read_csv(data_path)
else:
    df = pd.read_parquet(data_path)

non_glucose_features = [
    'hour_sin', 'hour_cos', 'sin_2phi', 'cos_2phi', 'is_night', 'day_of_week',
    'HR_unified', 'ECG_SDNN', 'ECG_RMSSD', 'DeviceTemp', 'Activity', 'BR',
    'HR_unified_roll15', 'ECG_SDNN_roll15', 'Activity_roll15', 'DeviceTemp_roll15'
]
non_glucose_features = [c for c in non_glucose_features if c in df.columns]

print(f'Strictly Non-Invasive Feature Set ({len(non_glucose_features)}):', non_glucose_features)
display(df[non_glucose_features + ['range_cgm', 'trend_30min', 'trend_60min']].head(5))

## Step 2 — Target Class Distributions & Clinical Justification
- **Range Targets:** Low ($<70$ mg/dL), Target ($70-180$ mg/dL), High ($>180$ mg/dL).
- **Trend Targets:** Decreasing ($<-1.0$ mg/dL/min), Stable ($-1.0$ to $+1.0$ mg/dL/min), Increasing ($>+1.0$ mg/dL/min).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['range_cgm'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#d62728', '#2ca02c', '#ff7f0e'])
axes[0].set_title('Glycemic Range Class Distribution', fontweight='bold')
axes[0].set_xticklabels(['Low (<70)', 'Target (70-180)', 'High (>180)'], rotation=0)
axes[0].set_ylabel('Reading Count')

df['trend_30min'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#1f77b4', '#7f7f7f', '#e377c2'])
axes[1].set_title('30-min Trend Direction Distribution', fontweight='bold')
axes[1].set_xticklabels(['Decreasing', 'Stable', 'Increasing'], rotation=0)
axes[1].set_ylabel('Reading Count')

plt.tight_layout()
plt.show()

## Step 3 — Leave-One-Subject-Out (LOSO) Classification Benchmark
We evaluate Majority Baseline, Logistic Regression, Random Forest, XGBoost, LightGBM, and PyTorch MLP classifiers.

In [ ]:
from phase2_noninvasive_loso import run_phase2_noninvasive_benchmark
print('Executing Full Phase 2 Non-Invasive Benchmark...')
run_phase2_noninvasive_benchmark()

## Step 4 — Comparative Results Summary

In [ ]:
p2_results = pd.read_csv('results/phase2_loso_results.csv')
display(p2_results.sort_values(['Task', 'Balanced_Accuracy'], ascending=[True, False]))

## Step 5 — Visualizing Confusion Matrices & Feature Importances

In [ ]:
from IPython.display import Image, display
display(Image('figures/phase2_balanced_accuracy_comparison.png'))
display(Image('figures/phase2_confusion_matrices.png'))
display(Image('figures/phase2_feature_importances.png'))

## Step 6 — Phase 2 Key Takeaways
1. **Non-Invasive Signal Viability:** Wearables alone achieve 42.7% balanced accuracy (Logistic Regression) / 41.2% (LightGBM) / 41.1% (Random Forest) for glycemic range classification and 39.6% for 30-min trend prediction under strict LOSO evaluation, substantially outperforming random chance (33.3%).
2. **Top Physiological Predictors:** Heart rate (HR_unified), circadian time harmonics (hour_sin, hour_cos), and raw ECG-derived HRV (ECG_SDNN, ECG_RMSSD) provided the strongest non-invasive discriminating power.
3. **Clinical Screening Utility:** While non-invasive sensing cannot replace continuous CGM for automated insulin titration, it offers high utility as a continuous passive prediabetes / glycemic anomaly screening tool on consumer smartwatches.